# ETL — 12 quyển / 3 NXB / 2 399 trang · cập nhật 2026-08-23 (lượt 3)

> ## ⛔ CHƯA CHẠY ĐƯỢC TOÀN CORPUS HÔM NAY — đọc hết ô này trước khi bấm Run all
>
> `--build-manifests` **bắt buộc** (manifest cũ đã bị xoá cùng `database/`) nhưng
> nó **chưa chạy được cho 8 quyển CTST + Cánh Diều**: bộ đọc MỤC LỤC chỉ quét đầu
> sách (`toc.TOC_SEARCH_PAGES = range(3, 10)`) trong khi MỤC LỤC của cả 4 quyển CD
> nằm ở **hai trang CUỐI**, và `toc._BAI` phân biệt hoa/thường nên `BÀI 1:` của
> CTST không khớp; CD thì không có chữ "Bài" nào (mục là `N. Tiêu đề <số>`).
> Chi tiết: `document/specs/2026-08-23-m0-report.md` §7. Đây là việc **M1**.
>
> **Chạy được ngay hôm nay:** 4 quyển KNTT (`--book SGK_KHTN_{6,7,8,9}_KNTT`).
> Đó là lượt chạy để **đo s/trang thật**, không phải lượt dựng DB cuối cùng.

**Nguồn:** `datasources/` là **12 thư mục PNG, một file mỗi trang**
(`SGK_KHTN_{6,7,8,9}_{KNTT,CTST,CD}/page_001.png …`), **2 399 trang**, 0 khoảng
trống, mọi quyển bắt đầu từ `page_001`.

```
CD   179 + 171 + 207 + 215 = 772 | CTST 204 + 188 + 223 + 215 = 830
KNTT 195 + 179 + 196 + 227 = 797                        tổng 2 399
```

**Năm thứ ĐÃ ĐỔI so với lượt 2 — mỗi cái làm hỏng một ô của notebook cũ:**

1. **`datasources/` KHÔNG còn trong git (D-68).** Bản clone chỉ có
   `datasources/README.md`. **Phải trỏ `RAG_DATA_DIR` sang Drive** (mục 5). Trỏ
   sai chỗ thì cả bốn entrypoint **thoát mã 2** kèm thông báo — không còn cảnh
   "chạy xong mà không xử lý gì".
2. **`--build-manifests` nay BẮT BUỘC** (ngược hẳn lượt 2). `database/manifests/`
   đã bị xoá cùng corpus cũ; đường text **raise `ManifestMissing`** nếu thiếu.
3. **`offset = 0`, không còn `−1` (D-65).** `printed_page == số trong tên file`,
   đo 40 trang/quyển trên 12/12 quyển. `page_001` **không** còn là bìa mặc định.
4. **KNTT là bộ ĐỘ PHÂN GIẢI THẤP NHẤT**, không phải bộ tham chiếu: KNTT
   1094×1536 vs CTST/CD **2280×3201** (6_CD 2480×3480) — ~3,5 lần diện tích. Nên
   **mọi con số s/trang trong notebook cũ chỉ đúng cho KNTT** và là **cận dưới**
   cho CD/CTST. Đừng hứa lịch theo chúng; đọc `s/trang` mà chính log ETL in ra.
5. **`RAG_FINGERPRINT_DIR` là biến MỚI** (mục 5). Fingerprint layout M0
   (`database/fingerprints/*.json`, 12/12 quyển) đã commit trong repo và đi theo
   **repo**, không theo Drive — cùng lý do như manifest: nó là kết quả đo, một
   lượt đo lại tốn ~70 phút OCR.

**Vẫn đúng từ lượt 2, đừng đổi:**

- **Caption ảnh TẮT** (`IMAGE_CAPTION_ENABLED=false`, D-47) — đo trên 12 crop
  thật: 17,6 s/crop trên CPU, **4/12 caption bịa** chi tiết không có trong ảnh,
  **0/4** lần tự nêu số hiệu hình là đúng. Bật lại mà không đo = tự bắn vào chân.
- **Checkpoint khoá theo hash TỪNG TRANG + version.** Sửa 1 trang → chỉ trang đó
  chạy lại; **bump version là cách DUY NHẤT** ép làm lại toàn bộ một phía.
- Text embedding **`BAAI/bge-m3`** (1024 chiều) + cross-encoder
  **`BAAI/bge-reranker-v2-m3`**. Đổi model embedding thì **phải dựng lại index**.
- Secret lấy từ **Colab Secrets** (🔑), không hardcode.

**Chi phí — ngoại suy, PHẢI đo lại trên một quyển CD trước khi hứa lịch:** text
3,56 s/trang × 2 399 ≈ **2,4 giờ**; ảnh 8,86 s/trang × 2 399 ≈ **5,9 giờ**. Cả hai
đo trên KNTT 1094×1536 nên với CD/CTST (3,5 lần diện tích) chúng là **cận dưới**.

> ⚠️ **Bảo mật:** notebook này từng hardcode `HF_TOKEN` trong ô mã (đã lộ vào git
> history). **Revoke token HF cũ + GitHub PAT cũ**, tạo token mới, lưu vào Colab
> Secrets tên `HF_TOKEN`.


## 1. Clone repo

In [ ]:
# Pipeline nguồn PNG đã merge vào master.
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git

In [ ]:
%cd project-bio-rag
!git log --oneline -3

## 2. Cài dependencies + Tesseract (vie)

`poppler-utils` chỉ còn cần cho đường upload PDF legacy (`/api/etl`) — nguồn PNG
không dùng. Cài luôn cho chắc, nhẹ.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-vie
!tesseract --version | head -1 && tesseract --list-langs | grep -x vie

## 3. Secret + env cơ bản (đặt TRƯỚC khi tải model)

`HF_TOKEN` lấy từ Colab Secrets. Mở tab 🔑 (Secrets) bên trái, thêm khoá
`HF_TOKEN`, bật *Notebook access*.

In [ ]:
import os, multiprocessing
from google.colab import userdata

# HF token từ Colab Secrets — KHÔNG hardcode (bản trước hardcode và đã làm lộ token).
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Đa luồng khớp số CPU thực tế
n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)

os.environ["USE_GPU"] = "true"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| CPU cores:", n)

## 4. Tải model về `./models` (chạy ONLINE)

Tải **cả 6 model là ~15 GB**. Chỉ tải thứ mình cần:

| profile | model | dùng cho |
|---|---|---|
| `text-etl` | bge-m3 (~2 GB) | `--text-only` |
| `image-etl` | CLIP + OWL-ViT + Vintern (~9 GB) | `--image-only` |
| `serve` | bge-m3 + reranker + Qwen-3B + CLIP (~12 GB) | `--api` |
| `all` | tất cả (~15 GB) | mặc định |

> **Caption đã tắt (D-47) nên `--image-only` KHÔNG cần Vintern-1B (~2 GB).** Profile
> `image-etl` vẫn tải nó để giữ đúng nghĩa "đủ cho phía ảnh nếu bật caption"; muốn
> nhẹ thì tải đúng hai model cần:
> `--only clip-vit-base-patch16,owlvit-base-patch32`.

> **`serve` bắt buộc có `bge-reranker-v2-m3`.** Thiếu nó thì với `HF_HUB_OFFLINE=1`,
> `RerankedRetriever` chỉ log một `warning` MỖI TRUY VẤN rồi rơi về xếp theo khoảng
> cách — `RERANK_ENABLED=true` mà thực chất không rerank. Cổng G3 in ra dòng
> `rerank: ...` để bạn thấy chuyện đó; đừng bỏ qua nó.

Trên **Colab free** hãy dùng `--profile text-etl` cho lượt đầu: nhanh hơn nhiều và
không ăn hết disk. `HF_HUB_OFFLINE` chưa bật ở bước này để tải được.

In [ ]:
# Luot ETL text: chi can bge-m3
!python ./src/utils/download_models.py --save_dir ./models --profile text-etl

# Phia anh - caption da tat nen chi can detector + CLIP (nhe hon profile image-etl):
# !python ./src/utils/download_models.py --save_dir ./models --only clip-vit-base-patch16,owlvit-base-patch32

# Serve API (BAT BUOC co reranker, xem ghi chu o tren):
# !python ./src/utils/download_models.py --save_dir ./models --profile serve

## 5. Mount Drive + trỏ DB ra Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Index nặng (ChromaDB + ảnh crop) -> Drive, bền qua các phiên Colab.
os.environ["RAG_DATABASE_DIR"] = "/content/drive/MyDrive/project_bio_rag/database"

# ĐỔI TỪ 2026-08-23 (D-68): `datasources/` KHÔNG còn trong git — bản clone chỉ có
# README.md. Ảnh trang PHẢI nằm trên Drive (hoặc copy về /content). Cấu trúc:
#   <RAG_DATA_DIR>/SGK_KHTN_6_KNTT/page_001.png ...
# Trỏ vào chỗ trống thì main.py thoát mã 2, không âm thầm chạy 0 quyển.
os.environ["RAG_DATA_DIR"] = "/content/drive/MyDrive/project_bio_rag/datasources"

# Manifest (bản đồ trang + spine Bài) đi theo REPO, không theo Drive.
os.environ["RAG_MANIFEST_DIR"] = "/content/project-bio-rag/database/manifests"

# MỚI: fingerprint layout M0 (12/12 quyển) cũng đi theo REPO — nó là KẾT QUẢ ĐO đã
# commit, đo lại tốn ~70 phút OCR. Không đặt biến này thì mặc định đã là
# <repo>/database/fingerprints (đường dẫn tuyệt đối, không phụ thuộc cwd — D-69).
os.environ["RAG_FINGERPRINT_DIR"] = "/content/project-bio-rag/database/fingerprints"

for key in ("RAG_DATABASE_DIR", "RAG_DATA_DIR", "RAG_MANIFEST_DIR",
            "RAG_FINGERPRINT_DIR"):
    print(f"{key} = {os.environ[key]}")

# Đọc ảnh 2280x3201 trực tiếp từ Drive rất chậm. Nếu chạy cả CD/CTST, copy về đĩa
# local của Colab trước rồi trỏ lại (bỏ comment):
# !mkdir -p /content/datasources && cp -r "$RAG_DATA_DIR"/* /content/datasources/
# os.environ["RAG_DATA_DIR"] = "/content/datasources"


### 5b. Kiểm tra nguồn trước khi chạy bất cứ thứ gì

Kỳ vọng (đo 2026-08-23, D-65) — **12 quyển, 2 399 trang, 0 khoảng trống**:

| NXB | số trang 6/7/8/9 | kích thước |
|---|---|---|
| KNTT | 195 / 179 / 196 / 227 = 797 | 1094×1536 (RGBA) |
| CTST | 204 / 188 / 223 / 215 = 830 | 2280×3201 (RGBA) |
| CD | 179 / 171 / 207 / 215 = 772 | 2280×3201; **6_CD là 2480×3480** (RGB) |

Ô dưới in `thieu: []` cho **mọi** quyển thì mới chạy tiếp. Thiếu quyển nào tức
`RAG_DATA_DIR` chưa trỏ đúng chỗ trên Drive — sửa mục 5, đừng chạy tiếp.


In [ ]:
import torch
from src.etl.page_source import discover_page_sources
import os

print("CUDA:", torch.cuda.is_available())
total = 0
for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
    numbers = source.page_numbers()
    gaps = [n for n in range(numbers[0], numbers[-1] + 1) if n not in set(numbers)]
    total += len(numbers)
    print(f"{source.name}: {len(numbers)} trang, {numbers[0]}..{numbers[-1]}, thieu: {gaps}")
print("TONG:", total, "| shape trang mau:", source.load(numbers[0]).shape)

> **(Tuỳ chọn) Build SẠCH từ đầu.** Checkpoint resume sẽ *bỏ qua* trang đã xử lý.
> Muốn dựng lại hoàn toàn, bỏ comment ô dưới (⚠️ **XOÁ toàn bộ index ở
> `RAG_DATABASE_DIR`**). Manifest KHÔNG bị xoá vì nó nằm trong repo
> (`RAG_MANIFEST_DIR`) — đó là điều mong muốn: bản đồ trang không cần dựng lại.
>
> Cách nhẹ hơn mà không xoá gì: **bump `TEXT_EXTRACTION_VERSION`** (mục 6) — mỗi
> trang sẽ được OCR lại và chunk cũ của chính trang đó bị xoá trước khi ghi mới.

In [ ]:
# import shutil, os
# shutil.rmtree(os.environ["RAG_DATABASE_DIR"], ignore_errors=True)
# os.makedirs(os.environ["RAG_DATABASE_DIR"], exist_ok=True)
# print("Đã xoá sạch:", os.environ["RAG_DATABASE_DIR"])

## 6. Env runtime — trỏ model local + bật offline + version gate

`HF_HUB_OFFLINE=1` (model đã tải ở bước 4). Hai biến version ở cuối ô là **cách
duy nhất** ép làm lại toàn bộ một phía; không đổi thì lượt chạy sau skip sạch (đó
là ý muốn, không phải lỗi).

In [ ]:
import os
base = "/content/project-bio-rag/models"

os.environ["HF_HUB_OFFLINE"] = "1"          # model da tai o buoc 4

# --- Text: bge-m3 + reranker ---
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_ENABLED"] = "true"
os.environ["RERANK_MODEL"]  = f"{base}/bge-reranker-v2-m3"

# --- LLM + image models ---
os.environ["LLM_MODEL"]     = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"]    = f"{base}/clip-vit-base-patch16"
os.environ["OWL_VIT_MODEL"] = f"{base}/owlvit-base-patch32"

# --- Caption anh: TAT (D-47) ---
# Da do tren 12 crop that: 17,6 s/crop tren CPU, 4/12 caption BIA chi tiet khong
# co trong anh, 0/4 lan model tu neu so hieu hinh la dung. Dung bat lai ma khong
# do lai. Bat ma model khong nap duoc thi ETL RAISE, khong nuot loi.
os.environ["IMAGE_CAPTION_MODEL"]   = f"{base}/Vintern-1B-v2"   # tro khi tat
os.environ["IMAGE_CAPTION_ENABLED"] = "false"

# --- Version gate: DOI GIA TRI = ep lam lai toan bo phia do ---
# Ca hai deu da bump so voi luot chay truoc, nen neu Drive con index cu thi ca
# hai phia se chay lai tu dau - dung y muon, vi ket qua hai phia deu da doi.
os.environ["TEXT_EXTRACTION_VERSION"]  = "v2_bai_spine"       # spine Bai -> bai_so vao metadata
os.environ["IMAGE_EXTRACTION_VERSION"] = "v19_pill_kernels"   # pill hop nhieu kernel CLOSE
print("text ver :", os.environ["TEXT_EXTRACTION_VERSION"])
print("image ver:", os.environ["IMAGE_EXTRACTION_VERSION"])
print("caption  :", os.environ["IMAGE_CAPTION_ENABLED"], "(tat = dung, xem D-47)")

## 7. BƯỚC 0 — `BookManifest` + cổng G1  ·  **BẮT BUỘC, ĐỪNG BỎ QUA**

Manifest là **nguồn sự thật duy nhất về số trang in**; đường text dừng với
`ManifestMissing` nếu thiếu, nó không đoán.

> ### Ngược hẳn lượt 2 của notebook này
> Lượt 2 ghi "manifest đã commit trong repo → bỏ qua ô này". Câu đó **hết hiệu
> lực**: `database/manifests/` đã bị xoá cùng corpus 4 quyển cũ, nên **không có
> manifest nào trong repo**. Phải chạy ô dưới, và **commit manifest mới** sau khi
> G1 PASS.

**Chạy được / chưa chạy được, nói theo phép đo (D-65):**

| NXB | `--build-manifests` | vì sao |
|---|---|---|
| KNTT (4 quyển) | **chạy được** | MỤC LỤC ở đầu sách `[4,5]`, mục dạng `Bài N`, spine đọc được 55/42/47/51 = 195 liền mạch |
| CTST (4 quyển) | **CHƯA** | bố cục MỤC LỤC hai cột; bộ đọc một cột cho ra spine sai (0/2/2/5) và bị gắn cờ `KHONG_dang_tin`. `toc._BAI` phân biệt hoa/thường nên `BÀI 1:` không khớp |
| Cánh Diều (4 quyển) | **CHƯA** | MỤC LỤC ở **hai trang CUỐI** nhưng `TOC_SEARCH_PAGES = range(3, 10)` chỉ quét đầu sách; mục là `N. Tiêu đề <số>`, **không có chữ "Bài"** |

Nên hôm nay chạy **từng quyển KNTT**:

    python main.py --build-manifests --book SGK_KHTN_6_KNTT

**Cái bẫy đắt nhất, đã đo trên 7_CTST:** chạy bộ đọc MỤC LỤC một cột lên bố cục
hai cột sinh ra **số SAI MÀ TRÔNG HỢP LÝ** (`Bài 1 → trang 144`, thật là trang 6),
rồi ràng buộc đơn điệu giết 31 Bài còn lại. Luật "bỏ entry chứ không đoán" chặn 31
số sai nhưng **không chặn số sai đầu tiên**. Vì vậy bộ đọc cell chỉ chạy khi
`entry_style == "bai"`, và spine không liền mạch bị gắn cờ `KHONG_dang_tin`.

Ba điều cần hiểu đúng khi đọc báo cáo G1 (vẫn đúng từ lượt 2):

- **MỤC LỤC dựng spine, huy hiệu Bài chỉ XÁC NHẬN** và không bao giờ ghi đè (D-44
  đảo ngược luật cũ "banner thắng"). Lệch thì ghi `banner_toc_mismatch`.
- **`0/k` ở huy hiệu của sách 7/8/9 là con số THẬT, không phải lỗi cấu hình** —
  lục giác màu đặc chữ trắng, đã thử ba cách đọc, vẫn chưa đọc được. Nó được in ra
  chứ không bị che.
- `bai_so` chỉ đi vào metadata chunk khi spine sạch; quyển bị cờ
  `bai_numbers_not_contiguous`/`spine_out_of_order` thì tự động thôi ghi.


In [ ]:
!python main.py --build-manifests

In [ ]:
# Xem nhanh manifest đã dựng (KHÔNG chạy lại OCR)
import glob, json, os

for path in sorted(glob.glob(os.path.join(os.environ["RAG_MANIFEST_DIR"], "*.json"))):
    m = json.load(open(path, encoding="utf-8"))
    covers = [p["page_index"] for p in m["pages"] if p["role"] == "cover"]
    unread = [p["page_index"] for p in m["pages"]
              if p["source"] != "ocr_confirmed" and p["role"] != "cover"]
    kinds = {}
    for flag in m["flags"]:
        kinds[flag["kind"]] = kinds.get(flag["kind"], 0) + 1
    print(f"{m['book_id']}: {m['n_pages']} trang | offset {m['page_offset']} | "
          f"bìa {covers} | trang có số mà KHÔNG đọc được: {unread} | Bài {len(m['bai'])}")
    print("   flags:", kinds)

## 8. ETL — TEXT (dùng được)

OCR theo **vùng layout** (không phải cả trang) → chunk → ChromaDB. Trang bìa
(`role="cover"`) bị bỏ qua ở bước chunk, **file nguồn không bị xoá**.

Resume theo TỪNG TRANG: Colab ngắt giữa đường thì chạy lại đúng lệnh này, nó chỉ
làm phần còn thiếu và **không nhân bản chunk**.

~1,6 s/trang → **~21 phút cho 801 trang** trên 1 luồng CPU (OCR không dùng GPU).

In [ ]:
!python main.py --text-only

In [ ]:
# Còn bao nhiêu trang chưa index? (biết có bị ngắt giữa đường không)
# Chạy trong subprocess để không giữ model embedding trong RAM của notebook.
import subprocess, sys, textwrap

script = textwrap.dedent("""
    import os
    from src.etl import ProcessingStatus
    from src.etl.page_source import discover_page_sources

    status = ProcessingStatus()
    for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
        print(source.name,
              "| text còn thiếu:", len(status.pages_needing_text(source)),
              "| ảnh còn thiếu:", len(status.pages_needing_images(source)))
""")
done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print(done.stdout or done.stderr[-2000:])

## 9. ETL — ẢNH (**dùng được** — cổng G4 = 100%)

Crop figure + index CLIP. `IMAGE_EXTRACTION_VERSION=v19_pill_kernels`.

**Bản notebook trước ghi "OUTPUT CHƯA TIN ĐƯỢC" — câu đó đã hết hiệu lực.**
Milestone M3 đã xong và có cổng đo:

    python -m src.test.qa_figures --all-books --bai-per-book 4

**Kết quả đo (16 Bài / 4 quyển):** nhãn hình có crop **72/72 = 100%**, **0 hình gán
sai Bài**. Đường đi: 88,6% → 90,0% → 95,7% → 97,1% → 98,6% → **100%**, với "gán sai
Bài" giữ 0 suốt.

Cổng này **không cần người dán nhãn từng trang**: `Hình A.B` nghĩa là hình thứ B của
**Bài A**, nên ghép với spine liền mạch là tự kiểm chứng được.

**Ba điều phải đọc đúng, đừng báo cáo quá lời:**

- Số "thiếu" là **CẬN DƯỚI**. Hình cuối của một Bài bị sót thì `max(B)` tụt theo và
  không ai biết. Chuyện này **đã xảy ra thật**: `Hình 2.5` sách 8 từng mất mà cổng
  không hề báo thiếu, chỉ lộ ra khi so DANH SÁCH trước/sau (D-51).
- **Crop to không đồng nghĩa crop sai.** Hai crop > 40% trang đã mở ra xem bằng mắt
  và đều đúng (`Hình 1.12` sách 9 thật sự là hình cả trang gồm 8 trang chiếu).
- **Còn chưa đọc được:** ba nhãn ô so sánh ("Thông tin liên lạc" / "Sản xuất" /
  "Giao thông vận tải") — pill nằm trên dải màu **cùng hệ màu** với nó (đo trên
  `page_010`: pill sat 82, dải tím sat 157). Không ngưỡng saturation, dải hue, hay
  kernel CLOSE nào tách được. Đây là lớp bài toán D-40 chưa giải.

**Caption ảnh TẮT** (mục 2 ở đầu notebook). Nên hình được index **không có
`caption_vi`** — đó là lựa chọn đã đo, không phải lỗi. Kênh tìm ảnh vẫn còn CLIP +
metadata + kênh khớp cụm từ có dấu.

QA thị giác khi cần nhìn bằng mắt:

    python -m src.test.test_image_extraction_full --book SGK_KHTN_9_KNTT --page 16

In [ ]:
!python main.py --image-only

In [ ]:
# Vòng review người cho metadata ảnh (ngữ nghĩa file JSON rất dễ hiểu sai —
# đọc README §6 trước khi dùng). Xoá dấu # để chạy.
# !python main.py --export-image-review database/review_images.json
# ... sửa file JSON ...
# !python main.py --apply-image-review database/review_images.json --review-user khoa

### 9b. (Thay thế) `--etl` = text + ảnh trong một lượt

Chỉ dùng khi bạn chấp nhận trạng thái của phía ảnh ở mục 9. Đường text bên trong
`--etl` **giống hệt** `--text-only`.

In [ ]:
# !python main.py --etl

### 9c. Lượt chạy thử THẬT đã kiểm chứng notebook này (2026-08-21)

Không phải mô tả kỳ vọng — đây là log của một lượt chạy thật trên **máy dev
(Windows, 16 core, KHÔNG CUDA)** với **corpus scratch 12 trang** và **DB riêng**,
đúng đường `main.py --etl` mà notebook này gọi:

```
RAG_DATA_DIR=<scratch>/scratch_corpus        # SGK_KHTN_6_KNTT/page_020..031.png (12 trang)
RAG_DATABASE_DIR=<scratch>/db_colab          # DB riêng, không đụng database/ của repo
RAG_MANIFEST_DIR=<repo>/database/manifests   # manifest ĐÃ COMMIT, không chạy --build-manifests
```

**Lượt 1 — chạy hết, không cần can thiệp gì (3 phút 24 giây):**

```
[SGK_KHTN_6_KNTT] Extracting images: 100%|##########| 12/12 [02:02<00:00, 10.22s/it]
[SGK_KHTN_6_KNTT] Extracted 17 images from 12 pages
Added 17 image metadata docs to ImageVectorDB
Added 17 visual image docs to ImageVectorDB
Completed: SGK_KHTN_6_KNTT
ETL (FULL) pipeline completed!
```

Không có dòng `caption model unavailable` — vì caption **tắt tường minh**, không
phải tắt âm thầm như trước (D-42 → D-47). `pill` đọc được nhãn hình trên đường đi,
ví dụ `[pill] 3 nhãn hình đọc được từ pill: ['Hình 10.3', 'Hình 10.1', 'Hình 10.2']`.

**Lượt 2 — chạy lại ĐÚNG lệnh đó: bỏ qua sạch (12 giây, gần hết là nạp model):**

```
VectorDB initialized with 76 existing chunks
Previously processed files: text=1, images=1
[SGK_KHTN_6_KNTT] Already processed for both text and images, skipping
ETL (FULL) pipeline completed!
```

Đây là bằng chứng cho câu "Colab ngắt giữa đường thì chạy lại đúng lệnh này".

**Lượt 3 — checkpoint khoá theo HASH NỘI DUNG trang, không theo tên file.** Ghi nội
dung của `page_040.png` lên `page_025.png` (md5 đổi `fbe5f29c…` → `8dff0f99…`) rồi
chạy lại:

```
[SGK_KHTN_6_KNTT] Extracted 2 images from 1 pages
[SGK_KHTN_6_KNTT] xoá 1 doc ảnh cũ của 1 trang trước khi ghi bản mới
[SGK_KHTN_6_KNTT] Added 2 images to ImageVectorDB
```

**Đúng 1 trang** chạy lại, 11 trang còn lại không bị đụng. Kiểm tra DB sau đó:
`text còn thiếu: []`, `ảnh còn thiếu: []`, và chunk text đúng một mục cho mỗi
`page_index` 20..31 — **không mồ côi, không nhân bản**.

> 🐞 **Lượt chạy thử này tìm ra một lỗi thật (D-52), và nó đã được sửa ngay.** Trước
> khi sửa, lượt 3 làm trang 25 có **3** doc ảnh (`Hình 8.1` cũ + `Hình 11.6`/`Hình
> 11.7` mới) vì đường ảnh **không xoá** doc cũ: `image_id` là hash của CROP nên crop
> đổi thì id đổi và doc cũ không bị upsert đè. Crop mồ côi vẫn tra ra được — học
> sinh có thể được trả về một hình không còn tồn tại trên trang. Chuyện này **sẽ
> xảy ra với MỌI trang** ở lần bump `IMAGE_EXTRACTION_VERSION` này nếu không sửa.
> Nay `ImageVectorDB.delete_page_documents` xoá đúng những trang sắp ghi lại, trên
> cả hai collection ảnh. Đó là lý do phải CHẠY THỬ chứ không chỉ đọc code.

## 10. (Tuỳ chọn) Eval — Recall@k / MRR / **cổng G3**

Ba nhóm script, hai loại:

**Không cần LLM** (chạy được ngay):

    python -m src.test.qa_citation_page      # cổng G3: trang được TRÍCH DẪN có chứa câu trả lời?
    python src/test/recall_at_k.py           # recall@k + MRR, base vs rerank

**Cần `EVAL_LLM_*`** (endpoint OpenAI-compatible bất kỳ):

    python src/test/generate_testsets.py --dry-run   # chọn trang, KHÔNG gọi LLM
    python src/test/generate_testsets.py             # 25 câu/quyển
    python -m src.test.qa_citation_page --judge      # LLM cứu ca deterministic loại
    python src/test/evaluator.py                     # P/R/MRR + LLM judge 1–5

**Bộ test 12 quyển cũ KHÔNG dùng được nữa** và đã dọn sang
`src/test/testsets/_archive_12books_2026_07/` (ngoài glob): cả hai khoá vàng của nó
đều không khớp metadata chunk hiện hành — `source_book` ghi `"SGK KHTN 6 KNTT.pdf"`
trong khi metadata là `"SGK_KHTN_6_KNTT"`, và `source_page` ghi số trong TÊN FILE
thay vì số trang IN (lệch 1). Bản mới lấy khoá vàng **thẳng từ metadata chunk thật**
nên khớp bởi cấu tạo (D-48).

> ⚠️ **Bộ test do LLM sinh, CHƯA có người duyệt** (`_generation_meta.json` ghi
> `human_reviewed: false`). Mọi báo cáo dùng số từ đây **phải nói rõ điều đó**. Nó
> là thước đo tương đối giữa các cấu hình (ablation), không phải chân lý.

> ⚠️ **`metrics.PAGE_TOLERANCE` = 0.** Chunk không bao giờ vắt qua hai trang, nên
> dung sai ±1 chỉ tính một chunk ở trang KHÁC là "trúng" → thổi recall. Nếu bạn so
> với con số cũ đo bằng ±1 thì đó là hai thước đo khác nhau.

In [ ]:
# Benchmark recall nhanh (base vs rerank, + MRR) — không gọi LLM
!python src/test/recall_at_k.py

## 11. (Tuỳ chọn) Serve API + Cloudflare tunnel để demo

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Backend
!nohup python main.py --api --port 5000 > backend.log 2>&1 &
# Tunnel công khai
!nohup cloudflared tunnel --url http://localhost:5000 > cf_backend.log 2>&1 &

In [ ]:
import time
time.sleep(8)
print('--- backend.log ---');
!tail -n 15 backend.log
print('--- public URL ---')
!grep -o 'https://[^ ]*trycloudflare.com' cf_backend.log | head -n 1

In [ ]:
# Dừng backend + tunnel + giải phóng port
!pkill -f main.py || true
!pkill -f cloudflared || true
!fuser -k 5000/tcp || true

## 12. (Tuỳ chọn) Sao lưu DB

DB đã nằm sẵn trên Drive (`RAG_DATABASE_DIR`) nên **không cần** zip/commit. Nếu muốn tải bản zip về máy:

In [ ]:
import os
src = os.environ["RAG_DATABASE_DIR"]
!zip -r -q /content/database_backup.zip "$src"
from google.colab import files
files.download('/content/database_backup.zip')